# 04 - Star Schema Preparation

Notebook này bắt đầu chuyển dữ liệu đã làm sạch sang mô hình **Fact + Dimension** để chuẩn bị đưa vào SQL Server và Power BI.

## Mục tiêu

- Tạo `DimProduct`
- Tạo `DimCustomer`
- Tạo `DimSite`
- Tạo `DimPlant`
- Tạo `DimWeek`
- Tạo `DimDate`
- Tạo `FactSales`
- Tạo `FactInventory`
- Xử lý Unknown Key
- Kiểm tra PK / FK / cardinality
- Kiểm tra row count và tổng measures trước/sau modeling
- Lưu các bảng star schema ra `data/processed/star_schema/`

## Nguyên tắc

- Không merge Classification trực tiếp vào `DimProduct`
- Không merge COGS / Retail Price trực tiếp vào FactSales ở bước này
- Không xóa duplicate-looking Sales rows
- Không xóa return/reversal
- Giữ business keys gốc để audit

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_DATA_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")
STAR_PATH = PROCESSED_PATH / "star_schema"

STAR_PATH.mkdir(parents=True, exist_ok=True)

print("STAR_PATH:", STAR_PATH)

STAR_PATH: ../data/processed/star_schema


# 1. Load dữ liệu đã chuẩn bị

In [2]:
sales_path = PROCESSED_PATH / "sales_clean.pkl"
product_path = PROCESSED_PATH / "dim_product.pkl"
distribution_path = PROCESSED_PATH / "dim_store_channel.pkl"

if not sales_path.exists():
    sales_path = PROCESSED_PATH / "sales_combined_raw.pkl"

sales = pd.read_pickle(sales_path)
product = pd.read_pickle(product_path)
distribution = pd.read_pickle(distribution_path)

print("Sales:", sales.shape)
print("Product:", product.shape)
print("Distribution:", distribution.shape)

Sales: (831966, 21)
Product: (94867, 25)
Distribution: (3405, 23)


# 2. Helper functions

In [3]:
def normalize_key(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

def add_unknown_row(df, row_dict, key_col):
    unknown_df = pd.DataFrame([row_dict])
    result = pd.concat([unknown_df, df], ignore_index=True)
    assert result[key_col].is_unique
    return result

# 3. Tạo DimProduct

`product_id` đã được xác nhận unique trong Product Master.

Ta tạo thêm `product_key` dạng surrogate key:

- `0` = UNKNOWN
- `1...N` = products hợp lệ

In [4]:
dim_product = product.copy()

dim_product["product_id"] = normalize_key(dim_product["product_id"])

# Bảo đảm 1 row / product_id
assert dim_product["product_id"].notna().all()
assert dim_product["product_id"].is_unique

dim_product = dim_product.reset_index(drop=True)
dim_product.insert(0, "product_key", np.arange(1, len(dim_product) + 1))

unknown_product = {
    col: pd.NA for col in dim_product.columns
}
unknown_product["product_key"] = 0
unknown_product["product_id"] = "UNKNOWN"

for col in [
    "brand_name",
    "product_group",
    "detail_product_group",
    "gender",
    "color",
    "size"
]:
    if col in unknown_product:
        unknown_product[col] = "UNKNOWN"

dim_product = add_unknown_row(
    dim_product,
    unknown_product,
    "product_key"
)

print(dim_product.shape)
dim_product.head()

(94868, 26)


,product_key,color,color_group,listing_price,price_group,gender,product_group,detail_product_group,shoe_product,size_group,size,age_group,activity_group,image_copyright,lifestyle_group,launch_season,mold_code,heel_height,code_lock,option,cost_price,product_id,product_style_color,product_style,brand_name,vendor_name
0,0,UNKNOWN,<NA>,<NA>,<NA>,UNKNOWN,UNKNOWN,UNKNOWN,<NA>,<NA>,UNKNOWN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,UNKNOWN,<NA>,<NA>,UNKNOWN,<NA>
1,1,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,38.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,80e1107e5bf74598baffea3a7b6073c5DEN38,80e1107e5bf74598baffea3a7b6073c5DEN,80e1107e5bf74598baffea3a7b6073c5,Brand1,vendor0
2,2,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,39.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,c8223e6133a64491a006dc0f95c2bfd9DEN39,c8223e6133a64491a006dc0f95c2bfd9DEN,c8223e6133a64491a006dc0f95c2bfd9,Brand1,vendor0
3,3,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,40.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,bec30e131ee04e49a4c87bc56f135b13DEN40,bec30e131ee04e49a4c87bc56f135b13DEN,bec30e131ee04e49a4c87bc56f135b13,Brand1,vendor0
4,4,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,41.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,3f4e265b0ac740e9b9edfd23e0ba1ca5DEN41,3f4e265b0ac740e9b9edfd23e0ba1ca5DEN,3f4e265b0ac740e9b9edfd23e0ba1ca5,Brand1,vendor0


In [5]:
product_key_map = dict(
    zip(
        dim_product["product_id"],
        dim_product["product_key"]
    )
)

print("DimProduct PK unique:", dim_product["product_key"].is_unique)
print("Unknown product key:", product_key_map.get("UNKNOWN"))

DimProduct PK unique: True
Unknown product key: 0


# 4. Tạo DimCustomer

Distribution có `customer_id` unique nên phù hợp để xây `DimCustomer`.

Lưu ý: `site_store` không được dùng làm PK của Customer.

In [6]:
dim_customer = distribution.copy()

dim_customer["customer_id"] = normalize_key(
    dim_customer["customer_id"]
)

assert dim_customer["customer_id"].notna().all()
assert dim_customer["customer_id"].is_unique

# Không cần store_channel_key cũ nữa
dim_customer = dim_customer.drop(
    columns=["store_channel_key"],
    errors="ignore"
)

dim_customer = dim_customer.reset_index(drop=True)
dim_customer.insert(
    0,
    "customer_key",
    np.arange(1, len(dim_customer) + 1)
)

unknown_customer = {
    col: pd.NA for col in dim_customer.columns
}
unknown_customer["customer_key"] = 0
unknown_customer["customer_id"] = "UNKNOWN"

for col in [
    "customer_name",
    "channel_id",
    "region",
    "city_level",
    "store_type",
    "b2b_b2c"
]:
    if col in unknown_customer:
        unknown_customer[col] = "UNKNOWN"

dim_customer = add_unknown_row(
    dim_customer,
    unknown_customer,
    "customer_key"
)

print(dim_customer.shape)
dim_customer.head()

(3406, 23)


,customer_key,site_store,b2b_b2c,channel_id,region,city_level,store_concept,trade_term,area_range,store_type,urbanization,branch_area,address_2,address_3,showroom_area,warehouse_area,start_month,start_year,end_month,end_year,note,customer_id,customer_name
0,0,<NA>,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,<NA>,<NA>,<NA>,UNKNOWN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,UNKNOWN,UNKNOWN
1,1,60000003,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Bình Tân,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,435043cf9,customer878
2,2,60000006,B2B,ST,KVMN,Cấp 2,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Tp. Biên Hoà,ĐNI,NaN,NaN,8,2010,NaN,NaN,NaN,db3c83bfa,customer904
3,3,60000007,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. 10,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,f9bd8e994,customer1213
4,4,60000008,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Gò Vấp,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,c6fb2b695,customer1792


In [7]:
customer_key_map = dict(
    zip(
        dim_customer["customer_id"],
        dim_customer["customer_key"]
    )
)

print("DimCustomer PK unique:", dim_customer["customer_key"].is_unique)
print("Unknown customer key:", customer_key_map.get("UNKNOWN"))

DimCustomer PK unique: True
Unknown customer key: 0


# 5. Tạo DimSite

`site_store` không unique trong Distribution vì một site có thể gắn với nhiều customer.

Do đó ở bước này ta tạo một `DimSite` **bảo thủ**:

- lấy tập hợp site từ Sales và Distribution
- chỉ dùng `site_id` làm business key
- chưa nhét các attribute như region/city nếu chưa chứng minh chúng single-valued theo site

In [8]:
sales_sites = set(
    normalize_key(sales["site"]).dropna()
)

distribution_sites = set(
    normalize_key(distribution["site_store"]).dropna()
)

all_sites = sorted(
    sales_sites | distribution_sites
)

dim_site = pd.DataFrame({
    "site_id": all_sites
})

dim_site.insert(
    0,
    "site_key",
    np.arange(1, len(dim_site) + 1)
)

unknown_site = {
    "site_key": 0,
    "site_id": "UNKNOWN"
}

dim_site = add_unknown_row(
    dim_site,
    unknown_site,
    "site_key"
)

print(dim_site.shape)
dim_site.head()

(3159, 2)


,site_key,site_id
0,0,UNKNOWN
1,1,1100
2,2,1101
3,3,1102
4,4,1103


In [9]:
site_key_map = dict(
    zip(
        dim_site["site_id"],
        dim_site["site_key"]
    )
)

print("DimSite PK unique:", dim_site["site_key"].is_unique)

DimSite PK unique: True


## 5.1 Kiểm tra các attribute nào thực sự ổn định theo site

Nếu một attribute có `max_unique_per_site = 1`, nó là ứng viên tốt để enrich `DimSite`.

In [10]:
candidate_site_attrs = [
    c for c in [
        "region",
        "city_level",
        "store_concept",
        "trade_term",
        "store_type",
        "urbanization",
        "branch_area"
    ]
    if c in distribution.columns
]

site_attr_report = []

dist_site_check = distribution.copy()
dist_site_check["site_id"] = normalize_key(
    dist_site_check["site_store"]
)

for col in candidate_site_attrs:
    nunique_per_site = (
        dist_site_check
        .groupby("site_id")[col]
        .nunique(dropna=True)
    )

    site_attr_report.append({
        "attribute": col,
        "max_unique_per_site": nunique_per_site.max(),
        "sites_with_conflict": (nunique_per_site > 1).sum()
    })

site_attr_report = pd.DataFrame(site_attr_report)
site_attr_report

,attribute,max_unique_per_site,sites_with_conflict
0,region,2,1
1,city_level,2,2
2,store_concept,3,1
3,trade_term,2,2
4,store_type,2,4
5,urbanization,3,2
6,branch_area,1,0


# 6. Tạo DimWeek

Sales dùng `week` dạng `YYYYWW`.

Do Master Calendar không cover giai đoạn Sales, ta tạo `DimWeek` trực tiếp từ Sales.

`week_start_date` chỉ được tạo khi Year-Week hợp lệ theo ISO week. Các value không parse được (ví dụ nếu có `202153`) vẫn được giữ nhưng gắn cờ `is_valid_iso_week = False`.

In [11]:
sales_week_values = (
    normalize_key(sales["week"])
    .dropna()
    .drop_duplicates()
    .sort_values()
)

dim_week = pd.DataFrame({
    "year_week": sales_week_values
})

dim_week["year"] = (
    dim_week["year_week"]
    .str[:4]
    .astype(int)
)

dim_week["week_number"] = (
    dim_week["year_week"]
    .str[-2:]
    .astype(int)
)

def iso_week_start(year, week):
    try:
        return pd.Timestamp.fromisocalendar(
            int(year),
            int(week),
            1
        )
    except Exception:
        return pd.NaT

dim_week["week_start_date"] = [
    iso_week_start(y, w)
    for y, w in zip(
        dim_week["year"],
        dim_week["week_number"]
    )
]

dim_week["week_end_date"] = (
    dim_week["week_start_date"]
    + pd.to_timedelta(6, unit="D")
)

dim_week["is_valid_iso_week"] = (
    dim_week["week_start_date"].notna()
)

dim_week = dim_week.reset_index(drop=True)
dim_week.insert(
    0,
    "week_key",
    np.arange(1, len(dim_week) + 1)
)

unknown_week = {
    "week_key": 0,
    "year_week": "UNKNOWN",
    "year": pd.NA,
    "week_number": pd.NA,
    "week_start_date": pd.NaT,
    "week_end_date": pd.NaT,
    "is_valid_iso_week": False
}

dim_week = add_unknown_row(
    dim_week,
    unknown_week,
    "week_key"
)

print(dim_week.shape)
dim_week.head(10)

(86, 7)


,week_key,year_week,year,week_number,week_start_date,week_end_date,is_valid_iso_week
0,0,UNKNOWN,<NA>,<NA>,NaT,NaT,False
1,1,202153,2021,53,NaT,NaT,False
2,2,202201,2022,1,2022-01-03,2022-01-09,True
3,3,202202,2022,2,2022-01-10,2022-01-16,True
4,4,202203,2022,3,2022-01-17,2022-01-23,True
5,5,202204,2022,4,2022-01-24,2022-01-30,True
6,6,202205,2022,5,2022-01-31,2022-02-06,True
7,7,202206,2022,6,2022-02-07,2022-02-13,True
8,8,202207,2022,7,2022-02-14,2022-02-20,True
9,9,202208,2022,8,2022-02-21,2022-02-27,True


In [12]:
print("Invalid ISO weeks:")
dim_week[
    (dim_week["week_key"] != 0)
    & (~dim_week["is_valid_iso_week"])
]

Invalid ISO weeks:


,week_key,year_week,year,week_number,week_start_date,week_end_date,is_valid_iso_week
1,1,202153,2021,53,NaT,NaT,False


In [13]:
week_key_map = dict(
    zip(
        dim_week["year_week"],
        dim_week["week_key"]
    )
)

# 7. Tạo DimDate

Inventory có snapshot date dạng `YYYYMMDD`.

Ta tạo Date Dimension daily để dùng cho FactInventory và dùng được về sau cho Power BI.

In [14]:
INVENTORY_PATH = RAW_DATA_PATH / "Inventory_snapshot_data"
inventory_files = sorted(INVENTORY_PATH.glob("*.xlsx"))

inventory_dates = []

for file in inventory_files:
    temp = pd.read_excel(
        file,
        usecols=lambda c: c in ["calendar_yeer_week"]
    )

    if "calendar_yeer_week" in temp.columns:
        parsed = pd.to_datetime(
            normalize_key(temp["calendar_yeer_week"]),
            format="%Y%m%d",
            errors="coerce"
        )
        inventory_dates.extend(parsed.dropna().tolist())

if len(inventory_dates) == 0:
    raise ValueError("Không parse được snapshot date từ Inventory.")

min_date = min(inventory_dates).normalize()
max_date = max(inventory_dates).normalize()

print("Inventory min date:", min_date)
print("Inventory max date:", max_date)

Inventory min date: 2022-01-31 00:00:00
Inventory max date: 2022-12-31 00:00:00


In [15]:
date_range = pd.date_range(
    min_date,
    max_date,
    freq="D"
)

dim_date = pd.DataFrame({
    "date": date_range
})

dim_date["date_key"] = (
    dim_date["date"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.day_name()
dim_date["iso_week"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["iso_year"] = dim_date["date"].dt.isocalendar().year.astype(int)

dim_date["year_week"] = (
    dim_date["iso_year"].astype(str)
    + dim_date["iso_week"].astype(str).str.zfill(2)
)

dim_date = dim_date[
    [
        "date_key",
        "date",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "day",
        "day_name",
        "iso_year",
        "iso_week",
        "year_week"
    ]
]

assert dim_date["date_key"].is_unique

dim_date.head()

,date_key,date,year,quarter,month_number,month_name,day,day_name,iso_year,iso_week,year_week
0,20220131,2022-01-31,2022,1,1,January,31,Monday,2022,5,202205
1,20220201,2022-02-01,2022,1,2,February,1,Tuesday,2022,5,202205
2,20220202,2022-02-02,2022,1,2,February,2,Wednesday,2022,5,202205
3,20220203,2022-02-03,2022,1,2,February,3,Thursday,2022,5,202205
4,20220204,2022-02-04,2022,1,2,February,4,Friday,2022,5,202205


# 8. Chuẩn bị FactSales

Grain được giữ nguyên như source:

> Một row trong `FactSales` tương ứng một source Sales record như dataset cung cấp.

Ta không giả định đây là invoice line vì không có transaction/invoice ID.

In [16]:
fact_sales = sales.copy()

fact_sales["product_id"] = normalize_key(
    fact_sales["product_id"]
)

fact_sales["customer_id"] = normalize_key(
    fact_sales["customer_id"]
).fillna("UNKNOWN")

fact_sales["site_id"] = normalize_key(
    fact_sales["site"]
).fillna("UNKNOWN")

fact_sales["year_week"] = normalize_key(
    fact_sales["week"]
).fillna("UNKNOWN")

fact_sales["product_key"] = (
    fact_sales["product_id"]
    .map(product_key_map)
    .fillna(0)
    .astype(int)
)

fact_sales["customer_key"] = (
    fact_sales["customer_id"]
    .map(customer_key_map)
    .fillna(0)
    .astype(int)
)

fact_sales["site_key"] = (
    fact_sales["site_id"]
    .map(site_key_map)
    .fillna(0)
    .astype(int)
)

fact_sales["week_key"] = (
    fact_sales["year_week"]
    .map(week_key_map)
    .fillna(0)
    .astype(int)
)

fact_sales.insert(
    0,
    "sales_fact_id",
    np.arange(1, len(fact_sales) + 1)
)

print(fact_sales.shape)
fact_sales.head()

(831966, 28)


,sales_fact_id,month,week,site,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,customer_id,product_id,source_file,year,month_number,month_date,profit,profit_margin,transaction_type,potential_duplicate,is_loss_making,site_id,year_week,product_key,customer_key,site_key,week_key
0,1,2022001,202201,1800,1800,Online,Online,ZF2,1,495720,729000,9847d4248,d77fdd34a14845db97837e059b0aca00TRG42,TT T01-2022_split_1.xlsx,2022,1,2022-01-01,233280,0.32,Sale,False,False,1800,202201,63054,675,243,2
1,2,2022001,202204,1116,1100,CHTT,Bán lẻ,FP,1,221000,325000,2384aef55,e485c0ab7b9b470cbddb80ea7367e734DEN40,TT T01-2022_split_1.xlsx,2022,1,2022-01-01,104000,0.32,Sale,False,False,1116,202204,62172,708,17,5
2,3,2022001,202201,1134,1100,CHTT,Bán lẻ,FP,1,255000,375000,20c3e0442,ac88f78262ee4b589bc93b106b67af1dDEN42,TT T01-2022_split_1.xlsx,2022,1,2022-01-01,120000,0.32,Sale,False,False,1134,202201,59630,734,35,2
3,4,2022001,202204,1612,1600,CHTT,Bán lẻ,FP,1,258400,380000,e8b42ff8f,920641c624934c4a8695347737f8f59dDEN35,TT T01-2022_split_1.xlsx,2022,1,2022-01-01,121600,0.32,Sale,False,False,1612,202204,68105,894,221,5
4,5,2022001,202202,1511,1500,CHTT,Bán lẻ,FP,1,272000,400000,b8d51499a,6764565f4bb141138af7d9cbf0905d0dHOL33,TT T01-2022_split_1.xlsx,2022,1,2022-01-01,128000,0.32,Sale,True,False,1511,202202,57744,846,185,3


## 8.1 Chọn cột FactSales

Giữ business keys để audit, cùng surrogate foreign keys để dùng trong database.

In [17]:
sales_fact_cols = [
    "sales_fact_id",

    # Foreign keys
    "product_key",
    "customer_key",
    "site_key",
    "week_key",

    # Business keys / lineage
    "product_id",
    "customer_id",
    "site_id",
    "year_week",
    "month",
    "week",
    "branch_id",
    "channel_id",
    "distribution_channel",
    "distribution_channel_code",

    # Measures
    "sold_quantity",
    "cost_price",
    "net_price"
]

for optional_col in [
    "profit",
    "profit_margin",
    "transaction_type",
    "potential_duplicate",
    "is_loss_making",
    "source_file",
    "year",
    "month_number",
    "month_date"
]:
    if optional_col in fact_sales.columns:
        sales_fact_cols.append(optional_col)

fact_sales = fact_sales[
    [c for c in sales_fact_cols if c in fact_sales.columns]
]

fact_sales.head()

,sales_fact_id,product_key,customer_key,site_key,week_key,product_id,customer_id,site_id,year_week,month,week,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,profit,profit_margin,transaction_type,potential_duplicate,is_loss_making,source_file,year,month_number,month_date
0,1,63054,675,243,2,d77fdd34a14845db97837e059b0aca00TRG42,9847d4248,1800,202201,2022001,202201,1800,Online,Online,ZF2,1,495720,729000,233280,0.32,Sale,False,False,TT T01-2022_split_1.xlsx,2022,1,2022-01-01
1,2,62172,708,17,5,e485c0ab7b9b470cbddb80ea7367e734DEN40,2384aef55,1116,202204,2022001,202204,1100,CHTT,Bán lẻ,FP,1,221000,325000,104000,0.32,Sale,False,False,TT T01-2022_split_1.xlsx,2022,1,2022-01-01
2,3,59630,734,35,2,ac88f78262ee4b589bc93b106b67af1dDEN42,20c3e0442,1134,202201,2022001,202201,1100,CHTT,Bán lẻ,FP,1,255000,375000,120000,0.32,Sale,False,False,TT T01-2022_split_1.xlsx,2022,1,2022-01-01
3,4,68105,894,221,5,920641c624934c4a8695347737f8f59dDEN35,e8b42ff8f,1612,202204,2022001,202204,1600,CHTT,Bán lẻ,FP,1,258400,380000,121600,0.32,Sale,False,False,TT T01-2022_split_1.xlsx,2022,1,2022-01-01
4,5,57744,846,185,3,6764565f4bb141138af7d9cbf0905d0dHOL33,b8d51499a,1511,202202,2022001,202202,1500,CHTT,Bán lẻ,FP,1,272000,400000,128000,0.32,Sale,True,False,TT T01-2022_split_1.xlsx,2022,1,2022-01-01


# 9. Validate FactSales foreign keys

In [18]:
sales_fk_validation = pd.Series({
    "rows": len(fact_sales),
    "unknown_product_key": (fact_sales["product_key"] == 0).sum(),
    "unknown_customer_key": (fact_sales["customer_key"] == 0).sum(),
    "unknown_site_key": (fact_sales["site_key"] == 0).sum(),
    "unknown_week_key": (fact_sales["week_key"] == 0).sum(),
    "unique_sales_fact_id": fact_sales["sales_fact_id"].nunique()
})

sales_fk_validation

rows                    831966
unknown_product_key          6
unknown_customer_key         1
unknown_site_key             0
unknown_week_key             0
unique_sales_fact_id    831966
dtype: int64

# 10. Validate FactSales measures trước/sau modeling

Modeling không được làm mất hoặc nhân dữ liệu.

In [19]:
sales_measure_validation = pd.DataFrame({
    "Before": [
        len(sales),
        sales["sold_quantity"].sum(),
        sales["cost_price"].sum(),
        sales["net_price"].sum()
    ],
    "After": [
        len(fact_sales),
        fact_sales["sold_quantity"].sum(),
        fact_sales["cost_price"].sum(),
        fact_sales["net_price"].sum()
    ]
}, index=[
    "rows",
    "sold_quantity",
    "cost_price",
    "net_price"
])

sales_measure_validation["Difference"] = (
    sales_measure_validation["After"]
    - sales_measure_validation["Before"]
)

sales_measure_validation

,Before,After,Difference
rows,831966,831966,0
sold_quantity,1174515,1174515,0
cost_price,254170428358,254170428358,0
net_price,332230025498,332230025498,0


# 11. Tạo DimPlant cho Inventory

In [20]:
plant_parts = []

for file in inventory_files:
    temp = pd.read_excel(
        file,
        usecols=lambda c: c in ["plant"]
    )
    plant_parts.append(temp)

all_plants = pd.concat(
    plant_parts,
    ignore_index=True
)

plant_values = sorted(
    normalize_key(all_plants["plant"])
    .dropna()
    .drop_duplicates()
)

dim_plant = pd.DataFrame({
    "plant_id": plant_values
})

dim_plant.insert(
    0,
    "plant_key",
    np.arange(1, len(dim_plant) + 1)
)

dim_plant = add_unknown_row(
    dim_plant,
    {
        "plant_key": 0,
        "plant_id": "UNKNOWN"
    },
    "plant_key"
)

plant_key_map = dict(
    zip(
        dim_plant["plant_id"],
        dim_plant["plant_key"]
    )
)

print(dim_plant.shape)
dim_plant.head()

(60, 2)


,plant_key,plant_id
0,0,UNKNOWN
1,1,1201
2,2,1202
3,3,1203
4,4,1204


# 12. Tạo FactInventory

Grain giữ theo source inventory snapshot:

> snapshot_date + plant + sloc + product

In [21]:
inventory_parts = []

for file in inventory_files:
    temp = pd.read_excel(file)
    temp["source_file"] = file.name
    inventory_parts.append(temp)

inventory = pd.concat(
    inventory_parts,
    ignore_index=True
)

print("Inventory combined:", inventory.shape)
inventory.head()

Inventory combined: (1367080, 10)


,Unnamed: 0,index,plant,calendar_year,calendar_yeer_week,sloc,quantity,total_amount,product_id,source_file
0,0,0,1201,2022,20220228,3000,19,0,1259098aaa8e447181f13903f84e5db1OOO35,28-02-2022_Ton Kho 1201 - 1210.xlsx
1,1,1,1201,2022,20220228,3000,41,0,39b38616e4d649ab9c3b7d04e82e079fOOO36,28-02-2022_Ton Kho 1201 - 1210.xlsx
2,2,2,1201,2022,20220228,3000,96,0,8db14e88898e40a392f80ed69c30e206OOO37,28-02-2022_Ton Kho 1201 - 1210.xlsx
3,3,3,1201,2022,20220228,3000,87,0,7c15de90afd343338f93c8f65a0d8380OOO38,28-02-2022_Ton Kho 1201 - 1210.xlsx
4,4,4,1201,2022,20220228,3000,104,0,6c04a173aec34690b242ed4e09367e96OOO39,28-02-2022_Ton Kho 1201 - 1210.xlsx


In [22]:
fact_inventory = inventory.copy()

fact_inventory = fact_inventory.drop(
    columns=["Unnamed: 0", "index"],
    errors="ignore"
)

fact_inventory["product_id"] = normalize_key(
    fact_inventory["product_id"]
)

fact_inventory["plant_id"] = normalize_key(
    fact_inventory["plant"]
)

fact_inventory["snapshot_date"] = pd.to_datetime(
    normalize_key(fact_inventory["calendar_yeer_week"]),
    format="%Y%m%d",
    errors="coerce"
)

fact_inventory["date_key"] = (
    fact_inventory["snapshot_date"]
    .dt.strftime("%Y%m%d")
)

fact_inventory["date_key"] = pd.to_numeric(
    fact_inventory["date_key"],
    errors="coerce"
).fillna(0).astype(int)

fact_inventory["product_key"] = (
    fact_inventory["product_id"]
    .map(product_key_map)
    .fillna(0)
    .astype(int)
)

fact_inventory["plant_key"] = (
    fact_inventory["plant_id"]
    .map(plant_key_map)
    .fillna(0)
    .astype(int)
)

fact_inventory.insert(
    0,
    "inventory_fact_id",
    np.arange(1, len(fact_inventory) + 1)
)

print(fact_inventory.shape)
fact_inventory.head()

(1367080, 14)


,inventory_fact_id,plant,calendar_year,calendar_yeer_week,sloc,quantity,total_amount,product_id,source_file,plant_id,snapshot_date,date_key,product_key,plant_key
0,1,1201,2022,20220228,3000,19,0,1259098aaa8e447181f13903f84e5db1OOO35,28-02-2022_Ton Kho 1201 - 1210.xlsx,1201,2022-02-28,20220228,74678,1
1,2,1201,2022,20220228,3000,41,0,39b38616e4d649ab9c3b7d04e82e079fOOO36,28-02-2022_Ton Kho 1201 - 1210.xlsx,1201,2022-02-28,20220228,74679,1
2,3,1201,2022,20220228,3000,96,0,8db14e88898e40a392f80ed69c30e206OOO37,28-02-2022_Ton Kho 1201 - 1210.xlsx,1201,2022-02-28,20220228,74680,1
3,4,1201,2022,20220228,3000,87,0,7c15de90afd343338f93c8f65a0d8380OOO38,28-02-2022_Ton Kho 1201 - 1210.xlsx,1201,2022-02-28,20220228,74681,1
4,5,1201,2022,20220228,3000,104,0,6c04a173aec34690b242ed4e09367e96OOO39,28-02-2022_Ton Kho 1201 - 1210.xlsx,1201,2022-02-28,20220228,74682,1


## 12.1 Chọn cột FactInventory

In [23]:
inventory_fact_cols = [
    "inventory_fact_id",

    # Foreign keys
    "product_key",
    "plant_key",
    "date_key",

    # Business keys / audit
    "product_id",
    "plant_id",
    "sloc",
    "calendar_year",
    "calendar_yeer_week",
    "snapshot_date",

    # Measures
    "quantity",
    "total_amount",

    # lineage
    "source_file"
]

fact_inventory = fact_inventory[
    [c for c in inventory_fact_cols if c in fact_inventory.columns]
]

fact_inventory.head()

,inventory_fact_id,product_key,plant_key,date_key,product_id,plant_id,sloc,calendar_year,calendar_yeer_week,snapshot_date,quantity,total_amount,source_file
0,1,74678,1,20220228,1259098aaa8e447181f13903f84e5db1OOO35,1201,3000,2022,20220228,2022-02-28,19,0,28-02-2022_Ton Kho 1201 - 1210.xlsx
1,2,74679,1,20220228,39b38616e4d649ab9c3b7d04e82e079fOOO36,1201,3000,2022,20220228,2022-02-28,41,0,28-02-2022_Ton Kho 1201 - 1210.xlsx
2,3,74680,1,20220228,8db14e88898e40a392f80ed69c30e206OOO37,1201,3000,2022,20220228,2022-02-28,96,0,28-02-2022_Ton Kho 1201 - 1210.xlsx
3,4,74681,1,20220228,7c15de90afd343338f93c8f65a0d8380OOO38,1201,3000,2022,20220228,2022-02-28,87,0,28-02-2022_Ton Kho 1201 - 1210.xlsx
4,5,74682,1,20220228,6c04a173aec34690b242ed4e09367e96OOO39,1201,3000,2022,20220228,2022-02-28,104,0,28-02-2022_Ton Kho 1201 - 1210.xlsx


# 13. Validate FactInventory foreign keys

In [24]:
date_key_set = set(dim_date["date_key"])

inventory_fk_validation = pd.Series({
    "rows": len(fact_inventory),
    "unknown_product_key":
        (fact_inventory["product_key"] == 0).sum(),
    "unknown_plant_key":
        (fact_inventory["plant_key"] == 0).sum(),
    "invalid_or_missing_date_key":
        (~fact_inventory["date_key"].isin(date_key_set)).sum(),
    "unique_inventory_fact_id":
        fact_inventory["inventory_fact_id"].nunique()
})

inventory_fk_validation

rows                           1367080
unknown_product_key                 29
unknown_plant_key                    0
invalid_or_missing_date_key          0
unique_inventory_fact_id       1367080
dtype: int64

# 14. Validate Inventory measures trước/sau modeling

In [25]:
inventory_measure_validation = pd.DataFrame({
    "Before": [
        len(inventory),
        inventory["quantity"].sum(),
        inventory["total_amount"].sum()
    ],
    "After": [
        len(fact_inventory),
        fact_inventory["quantity"].sum(),
        fact_inventory["total_amount"].sum()
    ]
}, index=[
    "rows",
    "quantity",
    "total_amount"
])

inventory_measure_validation["Difference"] = (
    inventory_measure_validation["After"]
    - inventory_measure_validation["Before"]
)

inventory_measure_validation

,Before,After,Difference
rows,1367080,1367080,0
quantity,2253264,2253264,0
total_amount,81552273459,81552273459,0


# 15. Cardinality checks

Điều kiện cơ bản của Star Schema:

- Dimension surrogate key phải unique
- Fact foreign key được phép lặp nhiều lần
- Fact FK phải tồn tại trong Dimension, hoặc map về Unknown Key = 0

In [26]:
cardinality_check = pd.DataFrame([
    {
        "dimension": "DimProduct",
        "key": "product_key",
        "rows": len(dim_product),
        "unique_keys": dim_product["product_key"].nunique(),
        "duplicate_keys": dim_product.duplicated("product_key").sum()
    },
    {
        "dimension": "DimCustomer",
        "key": "customer_key",
        "rows": len(dim_customer),
        "unique_keys": dim_customer["customer_key"].nunique(),
        "duplicate_keys": dim_customer.duplicated("customer_key").sum()
    },
    {
        "dimension": "DimSite",
        "key": "site_key",
        "rows": len(dim_site),
        "unique_keys": dim_site["site_key"].nunique(),
        "duplicate_keys": dim_site.duplicated("site_key").sum()
    },
    {
        "dimension": "DimWeek",
        "key": "week_key",
        "rows": len(dim_week),
        "unique_keys": dim_week["week_key"].nunique(),
        "duplicate_keys": dim_week.duplicated("week_key").sum()
    },
    {
        "dimension": "DimDate",
        "key": "date_key",
        "rows": len(dim_date),
        "unique_keys": dim_date["date_key"].nunique(),
        "duplicate_keys": dim_date.duplicated("date_key").sum()
    },
    {
        "dimension": "DimPlant",
        "key": "plant_key",
        "rows": len(dim_plant),
        "unique_keys": dim_plant["plant_key"].nunique(),
        "duplicate_keys": dim_plant.duplicated("plant_key").sum()
    }
])

cardinality_check

,dimension,key,rows,unique_keys,duplicate_keys
0,DimProduct,product_key,94868,94868,0
1,DimCustomer,customer_key,3406,3406,0
2,DimSite,site_key,3159,3159,0
3,DimWeek,week_key,86,86,0
4,DimDate,date_key,335,335,0
5,DimPlant,plant_key,60,60,0


# 16. Star Schema hiện tại

```text
                     DimProduct
                        │
                        │ 1:N
                        ▼
DimWeek ──────────── FactSales ─────────── DimCustomer
                        │
                        ▼
                     DimSite


                     DimProduct
                        │
                        │ 1:N
                        ▼
DimDate ────────── FactInventory ───────── DimPlant
```

### Chưa đưa vào model chính

- Classification: grain còn theo season/status
- COGS: effective-dated
- Retail Price: effective-dated

Các bảng đó sẽ được xử lý sau để tránh many-to-many và double counting.

# 17. Lưu Star Schema

In [27]:
outputs = {
    "dim_product.pkl": dim_product,
    "dim_customer.pkl": dim_customer,
    "dim_site.pkl": dim_site,
    "dim_plant.pkl": dim_plant,
    "dim_week.pkl": dim_week,
    "dim_date.pkl": dim_date,
    "fact_sales.pkl": fact_sales,
    "fact_inventory.pkl": fact_inventory
}

for filename, df in outputs.items():
    path = STAR_PATH / filename
    df.to_pickle(path)
    print(
        filename,
        "| rows =", len(df),
        "| size MB =",
        round(path.stat().st_size / 1024 / 1024, 2)
    )

dim_product.pkl | rows = 94868 | size MB = 20.85
dim_customer.pkl | rows = 3406 | size MB = 0.38
dim_site.pkl | rows = 3159 | size MB = 0.06
dim_plant.pkl | rows = 60 | size MB = 0.0
dim_week.pkl | rows = 86 | size MB = 0.0
dim_date.pkl | rows = 335 | size MB = 0.03
fact_sales.pkl | rows = 831966 | size MB = 151.88
fact_inventory.pkl | rows = 1367080 | size MB = 127.23


# 18. Final validation

In [28]:
final_star_validation = pd.Series({
    "dim_product_rows": len(dim_product),
    "dim_customer_rows": len(dim_customer),
    "dim_site_rows": len(dim_site),
    "dim_plant_rows": len(dim_plant),
    "dim_week_rows": len(dim_week),
    "dim_date_rows": len(dim_date),

    "fact_sales_rows": len(fact_sales),
    "fact_inventory_rows": len(fact_inventory),

    "sales_unknown_product":
        (fact_sales["product_key"] == 0).sum(),
    "sales_unknown_customer":
        (fact_sales["customer_key"] == 0).sum(),
    "sales_unknown_site":
        (fact_sales["site_key"] == 0).sum(),
    "sales_unknown_week":
        (fact_sales["week_key"] == 0).sum(),

    "inventory_unknown_product":
        (fact_inventory["product_key"] == 0).sum(),
    "inventory_unknown_plant":
        (fact_inventory["plant_key"] == 0).sum(),
    "inventory_invalid_date":
        (~fact_inventory["date_key"].isin(set(dim_date["date_key"]))).sum()
})

final_star_validation

dim_product_rows               94868
dim_customer_rows               3406
dim_site_rows                   3159
dim_plant_rows                    60
dim_week_rows                     86
dim_date_rows                    335
fact_sales_rows               831966
fact_inventory_rows          1367080
sales_unknown_product              6
sales_unknown_customer             1
sales_unknown_site                 0
sales_unknown_week                 0
inventory_unknown_product         29
inventory_unknown_plant            0
inventory_invalid_date             0
dtype: int64

# 19. Dừng tại đây

Sau khi **Run All → Save**, gửi lại notebook này.

Các output quan trọng cần kiểm tra:

- `site_attr_report`
- invalid ISO weeks trong `DimWeek`
- `sales_fk_validation`
- `sales_measure_validation`
- `inventory_fk_validation`
- `inventory_measure_validation`
- `cardinality_check`
- `final_star_validation`

Nếu row count và tổng measures trước/sau đều không đổi, và các Dimension keys đều unique, ta có thể chuyển sang bước tiếp theo:

## 05 - SQL Server Preparation

Ở bước đó sẽ tạo:

- SQL DDL
- Primary Key / Foreign Key
- bảng Fact / Dimension
- Python load script
- kiểm tra dữ liệu trong SQL
- chuẩn bị Power BI kết nối SQL Server